In [1]:
from pathlib import Path
import torch
from torch import nn as nn

In [2]:
torch.manual_seed(42)

x = torch.tensor([[-2.0], [-1.0], [0.0], [1.0], [2.0]],
                 dtype=torch.float32)
target = 2 * x + 1

In [3]:
model = nn.Linear(1,1)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(),lr = 0.05)

In [4]:
for step in range(1,51):
    optimizer.zero_grad(set_to_none=True)
    prediction = model(x)
    loss = criterion(prediction, target)
    loss.backward()
    optimizer.step()

    if step%10 == 0:
        with torch.no_grad():
            current_loss = criterion(model(x), target).item()
        print(
            f"step={step}, loss={current_loss:.6f}, "
            f"w={model.weight.item():.6f}, "
            f"b={model.bias.item():.6f}"
        )

step=10, loss=0.038709, w=1.867343, b=0.940727
step=20, loss=0.000833, w=1.985756, b=0.979333
step=30, loss=0.000057, w=1.998471, b=0.992794
step=40, loss=0.000006, w=1.999836, b=0.997487
step=50, loss=0.000001, w=1.999982, b=0.999124


In [5]:
model.eval()
with torch.no_grad():
    prediction_saved = model(x).clone()

print("保存前w/b:", model.weight.item(), model.bias.item())
print("保存前预测:", prediction_saved)
print("要保存的内容:", model.state_dict())

保存前w/b: 1.999982237815857 0.9991238713264465
保存前预测: tensor([[-3.0008],
        [-1.0009],
        [ 0.9991],
        [ 2.9991],
        [ 4.9991]])
要保存的内容: OrderedDict([('weight', tensor([[2.0000]])), ('bias', tensor([0.9991]))])


In [6]:
output_dir = Path(r"C:\\Users\\IPIC-ZQY\\Desktop\\codex\\AI-infra\\day7")
save_path = output_dir / "linear_day7_state.pth"

torch.save(model.state_dict(), save_path)

print("保存到:", save_path)
print("文件存在:", save_path.exists())

保存到: C:\Users\IPIC-ZQY\Desktop\codex\AI-infra\day7\linear_day7_state.pth
文件存在: True


In [7]:
restored_model = nn.Linear(1, 1)
restored_model.eval()

with torch.no_grad():
    prediction_before_load = restored_model(x).clone()

print("新模型加载前w/b:",
      restored_model.weight.item(), restored_model.bias.item())
print("新模型加载前预测:", prediction_before_load)

新模型加载前w/b: -0.23427248001098633 0.9186112880706787
新模型加载前预测: tensor([[1.3872],
        [1.1529],
        [0.9186],
        [0.6843],
        [0.4501]])


In [8]:
state = torch.load(save_path, map_location="cpu", weights_only=True)
restored_model.load_state_dict(state)
restored_model.eval()

with torch.no_grad():
    prediction_after_load = restored_model(x)

print("加载后w/b:",
      restored_model.weight.item(), restored_model.bias.item())
print("加载后预测:", prediction_after_load)

print("保存前与恢复后预测相近:",
      torch.allclose(prediction_saved, prediction_after_load))
print("最大绝对差:",
      (prediction_saved - prediction_after_load).abs().max().item())

加载后w/b: 1.999982237815857 0.9991238713264465
加载后预测: tensor([[-3.0008],
        [-1.0009],
        [ 0.9991],
        [ 2.9991],
        [ 4.9991]])
保存前与恢复后预测相近: True
最大绝对差: 0.0
